# Stage 14 check - preflight and smoke runs

Checks the shared folder and the runtime, then runs the data and training smoke
checks. Writes only to a `smoke_` folder and temporary directories.
Details: `docs/stage14_data_scale.md`.

## Setup

In [ ]:
# project folder on Drive; every account must see the shared folder at this path
PROJECT_DIR = '/content/drive/MyDrive/RAG chunk optimize'
ACCOUNT_LABEL = 'A'          # recorded in the run lock and the progress log
CLEAR_STALE_LOCK = False     # True only after confirming the runtime holding the lock is stopped

from google.colab import drive
drive.mount('/content/drive')

import os, shlex, subprocess, sys
if not os.path.isfile(os.path.join(PROJECT_DIR, 'config.py')):
    raise RuntimeError(f'no config.py under {PROJECT_DIR!r} - the shared folder is not mounted at this path.')
os.environ['RAG_DATA_ROOT'] = PROJECT_DIR + '/artifacts'
os.environ['PYTHONUNBUFFERED'] = '1'
sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)
GUARD = f' --account {ACCOUNT_LABEL}' + (' --clear-stale-lock' if CLEAR_STALE_LOCK else '')


def run(cmd):
    """Stream the command's output live; a failure stops Run All."""
    proc = subprocess.Popen(shlex.split(cmd), stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='', flush=True)
    if proc.wait() != 0:
        raise RuntimeError(f'command failed: {cmd}')

## Install dependencies

In [ ]:
run('pip install -q -r requirements.txt')

## Preflight

Shared-folder id, write access, GPU, run lock and progress so far.

In [ ]:
run('python -u scripts/35_preflight_stage14.py --gpu')

## Data smoke check

Six documents from the start of the stream, three shards of two, mined into a
`smoke_` folder.

In [ ]:
run('python -u scripts/32_build_stage14_data.py --smoke')

## Training smoke check

Eight Stage 8 groups, a checkpoint after every step, then the final save, all in a
temporary directory.

In [ ]:
run('python -u scripts/33_train_data_scale.py --smoke' + GUARD)